# Create multi-agent research tool using Autogen

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import List, Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
import requests
import xml.etree.ElementTree as ET



In [10]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

client = OpenAIChatCompletionClient(model="gpt-4o")

Implement an arXiv search function

In [13]:
def arxiv_search_tool(query: str, max_results: int) -> str:
    """
    Search for papers on arXiv based on the given query.

    Args:
        query (str): The search query.
        max_results (int): The maximum number of results to return.

    Returns:
        str: A list of the top results.
    """
    # Implementation for arXiv search would go here
    url = f'http://export.arxiv.org/api/query'
    response = requests.get(url, params={'search_query': query, 'max_results': max_results})

    if response.status_code == 200:
        root = ET.fromstring(response.content)
        entries = root.findall('{http://www.w3.org/2005/Atom}entry')
        results = []
        for entry in entries:
            title = entry.find('{http://www.w3.org/2005/Atom}title').text
            summary = entry.find('{http://www.w3.org/2005/Atom}summary').text
            results.append(f"Title: {title}\nSummary: {summary}\n")
        return "\n".join(results)
    else:
        return f"Error: Unable to fetch results from arXiv. Status code: {response.status_code}"
    

In [ ]:
result = arxiv_search_tool("Newton's laws of motion", 3)
print(result)

In [25]:
agent_description = "You are an agent that refines broad research topics."
SYSTEM_MESSAGE = "You are a research analyst. You take a general research idea and produce exactly one refined topic." \
"You should produce output in the following format, no additional text:" \
"Refined Research Topic: <Topic> End with: TASK_COMPLETED_TOPIC_AGENT"

topicAgent = AssistantAgent(name="TopicAgent",
                            description = agent_description,
                            system_message = SYSTEM_MESSAGE,
                            model_client = client
                            )

